# 大規模言語モデルの応用を体験する

## メディア研究とジャーナリズムのための 3 つの使い方

メディア研究を専攻する学部生のための第 2 回ハンズオンです。第 1 回では、Qwen3-8B の中間層から「感情ベクトル」を取り出し、LLM の内部表現を観察しました。

今回は視点を少し外側に移します。LLM を「研究対象」として見るだけでなく、メディア研究の作業を助ける「研究道具」として使うと、どんなことができるのでしょうか。

今日扱うのは次の 3 つです。

1. **代理ラベリング (surrogate labeling)**: YouTube 動画のタイトルと説明文から、気候変動否認論を含むかを判定する
2. **社会シミュレーション (social simulation)**: LLM エージェントを小さなソーシャルメディア社会として動かし、専門家介入や誤情報の影響を見る
3. **メカニズム分析 (mechanism analysis)**: 気候合意と気候否認の主張が、モデル内部のどのような幾何として見えるかを観察する

このノートブックは研究論文レベルの完全な実装ではありません。目的は、皆さんが「LLM を使ったメディア研究では、こういう問いの立て方ができるのか」と直感的に理解することです。コードセルは Colab では折りたたまれて表示されます。まずは文章と図を追ってください。


## ステップ 0: 環境セットアップ

第 1 回と同じく、Google Colab の無料 T4 GPU を想定します。

上部メニューから **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選んでください。下のセルはライブラリを入れ、データを読み込むための準備をします。


In [ ]:
#@title 環境セットアップ (実行してください) {display-mode: "form"}
import subprocess, sys, os, gc, json, re, math, random, urllib.request
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any

IN_COLAB = "google.colab" in sys.modules

PACKAGES = [
    "transformers>=4.51.0,<5.0.0",
    "huggingface_hub<1.0.0",
    "tokenizers>=0.21.0,<0.22.0",
    "accelerate",
    "bitsandbytes",
    "sentencepiece",
    "pandas",
    "numpy",
    "scikit-learn",
    "matplotlib",
    "japanize-matplotlib",
    "networkx",
    "tqdm",
    "openai>=1.0.0",
]

if IN_COLAB:
    print("Colab を検出しました。必要なライブラリをインストールしています...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *PACKAGES])
    print("インストール完了")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import networkx as nx
from tqdm.auto import tqdm
from IPython.display import display, Markdown
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

try:
    import japanize_matplotlib
except Exception:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_DIR = Path("/content/llmintro_app_cache") if IN_COLAB else Path.cwd() / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

GITHUB_DATA_BASE = "https://raw.githubusercontent.com/CSS-Laboratory/LLMintro/main/data/application"

def load_application_csv(filename: str) -> pd.DataFrame:
    """Load teaching data locally if available, otherwise from GitHub raw URL."""
    local_candidates = [
        Path.cwd() / "data" / "application" / filename,
        Path.cwd() / filename,
        CACHE_DIR / filename,
    ]
    for path in local_candidates:
        if path.exists():
            return pd.read_csv(path)

    url = f"{GITHUB_DATA_BASE}/{filename}"
    out = CACHE_DIR / filename
    print(f"GitHub からダウンロードします: {url}")
    urllib.request.urlretrieve(url, out)
    return pd.read_csv(out)

print(f"デバイス: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU が見つかりません。Colab では T4 GPU を選んでください。")


## 実行モード

このノートブックには、時間を短くするための設定があります。

- `LIVE_MODE = True`: Qwen3-8B を実際に読み込んで動かします
- `FAST_DEMO = True`: 学生向けの短いデモ設定です。サンプル数やシミュレーション日数を少なめにします
- `USE_QWEN_FOR_SOCIAL_SIM = True`: 社会シミュレーション内のエージェント更新にも Qwen3-8B を使います

授業で時間が足りない場合は `LIVE_MODE = False` にすると、モデルを使う重い部分は簡易デモに切り替わります。


In [ ]:
#@title 実行モードを設定 {display-mode: "form"}
LIVE_MODE = True  #@param {type:"boolean"}
FAST_DEMO = True  #@param {type:"boolean"}
USE_QWEN_FOR_SOCIAL_SIM = True  #@param {type:"boolean"}

print(f"LIVE_MODE: {LIVE_MODE}")
print(f"FAST_DEMO: {FAST_DEMO}")
print(f"USE_QWEN_FOR_SOCIAL_SIM: {USE_QWEN_FOR_SOCIAL_SIM}")


## Qwen3-8B を読み込む

第 1 回と同じモデルを使います。4-bit 量子化により T4 GPU でも動きます。

ここで重要なのは、今日のモデルは「正解表を持った採点者」ではなく、**次に来るトークンの確率分布を計算する機械**だという点です。代理ラベリングでも、社会シミュレーションでも、メカニズム分析でも、この性質がずっと効いてきます。


In [ ]:
#@title Qwen3-8B を 4-bit 量子化で読み込む (数分) {display-mode: "form"}
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-8B"

if LIVE_MODE:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR / "models")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        cache_dir=CACHE_DIR / "models",
        quantization_config=quant_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    NUM_LAYERS = len(model.model.layers)
    HIDDEN_SIZE = model.config.hidden_size
    LAYER_TO_PROBE = int(round(NUM_LAYERS * 2 / 3))

    print("モデル読み込み完了")
    print(f"Transformer 層: {NUM_LAYERS}")
    print(f"隠れ状態の次元: {HIDDEN_SIZE}")
    print(f"今日観察する層: {LAYER_TO_PROBE}")
else:
    tokenizer = None
    model = None
    NUM_LAYERS = 36
    HIDDEN_SIZE = 4096
    LAYER_TO_PROBE = 24
    print("デモモード: モデルは読み込みません。")


In [ ]:
#@title Qwen3-8B を使うための小さな関数 {display-mode: "form"}
def render_chat(user_prompt: str, system_prompt: Optional[str] = None) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def qwen_generate(
    user_prompt: str,
    system_prompt: Optional[str] = None,
    max_new_tokens: int = 80,
    temperature: float = 0.7,
    top_p: float = 0.9,
) -> str:
    if model is None:
        raise RuntimeError("LIVE_MODE=False のため Qwen 生成は使えません。")
    rendered = render_chat(user_prompt, system_prompt)
    inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def token_id_for_label(label: str) -> int:
    """Find a single-token id for a short label such as 0 or 1."""
    candidates = [label, " " + label]
    for cand in candidates:
        ids = tokenizer(cand, add_special_tokens=False)["input_ids"]
        if len(ids) == 1:
            return ids[0]
    raise ValueError(f"ラベル {label!r} が 1 トークンになりませんでした: {candidates}")


def next_token_probs_for_labels(
    user_prompt: str,
    labels: Tuple[str, ...] = ("0", "1"),
    system_prompt: Optional[str] = None,
) -> Dict[str, float]:
    """Constrain a classification task to the next token and read label probabilities."""
    if model is None:
        raise RuntimeError("LIVE_MODE=False のため logit は読めません。")
    rendered = render_chat(user_prompt, system_prompt)
    inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False).to(model.device)
    label_ids = [token_id_for_label(label) for label in labels]
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1, label_ids].float().cpu()
    probs = torch.softmax(logits, dim=0).numpy()
    return {label: float(prob) for label, prob in zip(labels, probs)}


def parse_binary_label(text: str) -> Optional[int]:
    match = re.search(r"\b([01])\b", str(text))
    return int(match.group(1)) if match else None


def parse_json_object(text: str) -> Optional[dict]:
    match = re.search(r"\{.*\}", str(text), flags=re.S)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None

print("補助関数を定義しました。")


---

# セクション 1: 代理ラベリング

## 直感: 1 人のアルバイトに 1 回だけ聞くのは危ない

内容分析では、人間のコーダーがコードブックを読んで、記事、投稿、動画などにラベルを付けます。LLM を使うと、この作業の一部を自動化できます。これをここでは **代理ラベリング** と呼びます。

ただし、LLM の出力は「答えそのもの」ではありません。モデルは、プロンプトを読んだあとに次のトークンの確率分布を作り、そこから 1 つの文字列を出します。

つまり、同じ動画に対して 1 回だけ `0` や `1` を出させると、確率分布の一部をたまたま見ているだけかもしれません。

研究で使うなら、少なくとも次のどちらかを考えます。

1. **複数回聞く**: 同じ対象を何度か分類させ、分布や不一致を見る
2. **logit を読む**: `0` と `1` の次トークン確率を直接比べ、モデルの迷いを見る


## データとコードブック

参照資料にある YouTube 動画サンプルと、気候変動否認論のコードブックを使います。

ラベル `denialism` は、人間コーディングによる参照ラベルです。`0` は否認論なし、`1` は否認論ありを意味します。中間値は、コーダー間の不一致や部分的な該当を表すものとして扱います。


In [ ]:
#@title YouTube サンプルとコードブックを読み込む {display-mode: "form"}
codebook_df = load_application_csv("climate_denial_codebook.csv")
sample_df = load_application_csv("climate_youtube_sample.csv")

print("コードブック:", codebook_df.shape)
print("YouTube サンプル:", sample_df.shape)
print("denialism の分布")
print(sample_df["denialism"].value_counts(dropna=False).sort_index())

display(codebook_df.head(10))
display(sample_df[["title", "channelTitle", "denialism", "conspiracy"]].head(8))


In [ ]:
#@title ラベリング用プロンプトを作る {display-mode: "form"}
def build_codebook_text(df: pd.DataFrame) -> str:
    lines = []
    for _, row in df.iterrows():
        parent = "" if pd.isna(row.get("Parent")) else f" (parent: {row['Parent']})"
        lines.append(f"- {row['Name']}{parent}: {row['Explanation']}")
    return "\n".join(lines)

CODEBOOK_TEXT = build_codebook_text(codebook_df)

LABEL_SYSTEM_PROMPT = """You are assisting a media studies content analysis project. Follow the codebook exactly. If the evidence is insufficient, choose 0."""


def build_denialism_prompt(row: pd.Series) -> str:
    title = str(row.get("title", ""))[:500]
    description = str(row.get("description", ""))[:1800]
    return f"""
以下の YouTube 動画が、気候変動否認論を含むかを判定してください。

コードブック:
{CODEBOOK_TEXT}

判定規則:
- 1 = タイトルまたは説明文に、コードブックのいずれかに該当する気候変動否認論が含まれる
- 0 = 気候変動について述べていても、否認論ではない。または情報が足りない
- 出力は 0 または 1 の 1 文字だけ

タイトル:
{title}

説明文:
{description}

ラベル:
""".strip()

print(build_denialism_prompt(sample_df.iloc[0])[:1200])


## 1 回だけ聞いたラベルを見る

まずは一番単純な方法を試します。LLM にコードブックと動画情報を渡し、`0` か `1` を 1 回だけ返してもらいます。

これは便利ですが、研究データとして使うには危険です。なぜなら、生成は確率的で、プロンプトや温度設定によって揺れることがあるからです。


In [ ]:
#@title 1 件だけ、1 回だけラベルを生成する {display-mode: "form"}
# 人間ラベルが中間値に近いものを、曖昧な例として選ぶ
ambiguous_idx = int((sample_df["denialism"].fillna(0) - 0.5).abs().sort_values().index[0])
row = sample_df.loc[ambiguous_idx]

print("対象タイトル:")
print(row["title"])
print(f"人間ラベル denialism: {row['denialism']}")
print()

if LIVE_MODE:
    raw = qwen_generate(
        build_denialism_prompt(row),
        system_prompt=LABEL_SYSTEM_PROMPT,
        max_new_tokens=8,
        temperature=0.7,
    )
    print("Qwen の出力:", repr(raw))
    print("パースしたラベル:", parse_binary_label(raw))
else:
    print("デモモード: ここでは実生成をスキップします。")


## 同じ対象を複数回聞く

同じ動画に複数回ラベルを付けさせると、モデルがどれくらい迷っているかが見えます。

たとえば 7 回中 7 回 `1` なら、少なくともこのプロンプトでは強く否認論だと見ています。7 回中 `0` と `1` が割れるなら、その対象はコードブック上も曖昧か、プロンプトが十分に明確でない可能性があります。


In [ ]:
#@title 同じ動画を複数回ラベリングする {display-mode: "form"}
N_REPEATS = 5 if FAST_DEMO else 9

if LIVE_MODE:
    repeated = []
    for i in tqdm(range(N_REPEATS), desc="同じ対象を複数回分類"):
        raw = qwen_generate(
            build_denialism_prompt(row),
            system_prompt=LABEL_SYSTEM_PROMPT,
            max_new_tokens=8,
            temperature=0.9,
        )
        repeated.append({"trial": i + 1, "raw_output": raw, "label": parse_binary_label(raw)})
    repeated_df = pd.DataFrame(repeated)
else:
    rng = np.random.default_rng(SEED)
    repeated_df = pd.DataFrame({
        "trial": range(1, N_REPEATS + 1),
        "raw_output": rng.choice(["0", "1"], size=N_REPEATS, p=[0.35, 0.65]),
    })
    repeated_df["label"] = repeated_df["raw_output"].astype(int)

display(repeated_df)
print("平均ラベル =", repeated_df["label"].dropna().mean())


## logit から確率を見る

次に、`0` と `1` を実際に生成させるのではなく、最後の位置でモデルが `0` と `1` にどのくらいの確率を置いているかを読みます。

これは「モデルの心の中を完全に読む」ことではありませんが、1 回の生成結果よりは、分類の不確実性を扱いやすくなります。


In [ ]:
#@title 0/1 の次トークン確率を読む {display-mode: "form"}
if LIVE_MODE:
    probs = next_token_probs_for_labels(
        build_denialism_prompt(row),
        labels=("0", "1"),
        system_prompt=LABEL_SYSTEM_PROMPT,
    )
else:
    probs = {"0": 0.32, "1": 0.68}

print("次トークン確率")
for label, prob in probs.items():
    print(f"  P({label}) = {prob:.3f}")
print("モデルが見ている denialism 確率の近似:", f"{probs['1']:.3f}")


## 小さなサンプル全体を分類する

授業用なので、全部ではなく一部だけ分類します。ここでは `P(1)` を使い、0.5 以上なら「否認論あり」とします。

研究で本当に使うなら、しきい値は固定でよいとは限りません。人間ラベルとの比較、コーダー間一致、検証用データでの較正が必要です。


In [ ]:
#@title 小さなサンプルを logit で分類する {display-mode: "form"}
N_POS = 4 if FAST_DEMO else 8
N_NEG = 6 if FAST_DEMO else 12

positive = sample_df[sample_df["denialism"].fillna(0) > 0].head(N_POS)
negative = sample_df[sample_df["denialism"].fillna(0) == 0].head(N_NEG)
label_subset = pd.concat([positive, negative], ignore_index=True)

records = []
if LIVE_MODE:
    for _, item in tqdm(label_subset.iterrows(), total=len(label_subset), desc="logit 分類"):
        probs = next_token_probs_for_labels(
            build_denialism_prompt(item),
            labels=("0", "1"),
            system_prompt=LABEL_SYSTEM_PROMPT,
        )
        p1 = probs["1"]
        records.append({
            "title": item["title"],
            "human": int(float(item["denialism"]) >= 0.5),
            "p_denialism": p1,
            "model": int(p1 >= 0.5),
        })
else:
    rng = np.random.default_rng(SEED)
    for _, item in label_subset.iterrows():
        human = int(float(item["denialism"]) >= 0.5)
        p1 = float(np.clip(0.25 + 0.55 * human + rng.normal(0, 0.15), 0.02, 0.98))
        records.append({"title": item["title"], "human": human, "p_denialism": p1, "model": int(p1 >= 0.5)})

label_results = pd.DataFrame(records)
display(label_results[["title", "human", "p_denialism", "model"]])

acc = accuracy_score(label_results["human"], label_results["model"])
cm = confusion_matrix(label_results["human"], label_results["model"], labels=[0, 1])
print(f"Accuracy: {acc:.2f}")
print("Confusion matrix rows=human, cols=model [0,1]:")
print(cm)


In [ ]:
#@title 図 1: 人間ラベルとモデル確率の比較 {display-mode: "form"}
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#4C78A8" if h == 0 else "#F58518" for h in label_results["human"]]
ax.scatter(range(len(label_results)), label_results["p_denialism"], c=colors, s=80, alpha=0.85)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1)
ax.set_ylim(-0.03, 1.03)
ax.set_xlabel("動画サンプル")
ax.set_ylabel("Qwen が置いた P(denialism=1)")
ax.set_title("1 回のラベルではなく、確率として見る")
ax.grid(alpha=0.25)
plt.show()


## OpenAI API のスモークテスト

全員が API キーを持っているとは限らないので、ここでは実際の API 呼び出しはデフォルトで行いません。

ただし、研究現場では次のように、API にコードブック、対象テキスト、構造化出力の指定を渡してラベリングできます。OpenAI の現在の推奨は Responses API です。下のセルは `DRY RUN` として、送るリクエストの形だけを表示します。


In [ ]:
#@title OpenAI API スモークテスト (デフォルトでは送信しない) {display-mode: "form"}
RUN_OPENAI_SMOKE_TEST = False  #@param {type:"boolean"}
OPENAI_MODEL = "gpt-5.5"  #@param {type:"string"}

from pprint import pprint

smoke_row = sample_df.iloc[0]
smoke_input = build_denialism_prompt(smoke_row)

request_payload = {
    "model": OPENAI_MODEL,
    "instructions": "You are assisting a media studies content analysis project. Follow the codebook exactly.",
    "input": [
        {"role": "user", "content": smoke_input}
    ],
    "text": {
        "format": {
            "type": "json_schema",
            "name": "climate_denialism_label",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "denialism": {"type": "integer", "enum": [0, 1]},
                    "confidence": {"type": "number", "minimum": 0, "maximum": 1},
                    "evidence": {"type": "string"},
                },
                "required": ["denialism", "confidence", "evidence"],
                "additionalProperties": False,
            },
        }
    },
}

if RUN_OPENAI_SMOKE_TEST:
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY が環境変数にありません。Colab Secrets を使ってください。")
    from openai import OpenAI
    client = OpenAI()
    response = client.responses.create(**request_payload)
    print(response.output_text)
else:
    print("DRY RUN: 実際の API 呼び出しは行っていません。送信するならこの形です。")
    pprint(request_payload, width=100)


## セクション 1 のまとめ

代理ラベリングは、メディア研究で非常に強力です。大量のニュース、投稿、動画説明文を、人間だけで読むより速く処理できます。

しかし、LLM ラベルを「人間ラベルの完全な代用品」と見なすのは危険です。

大事な実務上の考え方は次の 3 つです。

1. 1 回の返答ではなく、**確率や複数回の揺れ**を見る
2. コードブックを明確にし、人間ラベルで**較正**する
3. モデルが間違えやすい境界事例を、研究者が最後に確認する


---

# セクション 2: 社会シミュレーション

## 直感: 小さな人工社会で「もしも」を試す

社会シミュレーションでは、1 人ひとりのユーザーをエージェントとして表し、つながり、信念、記憶、役割を持たせます。

FUSE という研究枠組みは、LLM エージェントを使って「真実のニュースが、人々の相互作用を通じてどのように歪んでいくか」を調べます。重要な部品は次の 4 つです。

1. **役割**: 拡散者、コメントする人、検証者、傍観者
2. **ネットワーク**: 誰が誰の投稿を見るか
3. **記憶と更新**: 今日見た情報が、明日の理解に残る
4. **評価**: 元のニュースからどれくらいずれたかを測る

ここでは FUSE の完全再現ではなく、授業用の小さな FUSE 風モデルを作ります。


In [ ]:
#@title 小さな FUSE 風シミュレーションの部品 {display-mode: "form"}
@dataclass
class MiniAgent:
    id: int
    role: str
    belief: float  # 0 = climate consensus, 1 = strong climate denialism
    receptiveness: float
    message: str = ""

ROLE_EFFECT = {
    "spreader": 1.15,
    "commentator": 1.05,
    "verifier": 0.75,
    "bystander": 0.55,
}


def create_mini_agents(n: int, seed: int = SEED) -> List[MiniAgent]:
    rng = np.random.default_rng(seed)
    low = rng.beta(2, 8, size=n // 2)
    high = rng.beta(8, 2, size=n - n // 2)
    beliefs = np.clip(np.concatenate([low, high]), 0, 1)
    rng.shuffle(beliefs)
    roles = rng.choice(["spreader", "commentator", "verifier", "bystander"], size=n, p=[0.30, 0.45, 0.15, 0.10])
    agents = []
    for i, (belief, role) in enumerate(zip(beliefs, roles)):
        receptiveness = float(np.clip(0.85 - abs(belief - 0.5) + rng.normal(0, 0.08), 0.15, 0.95))
        message = "Climate change should be discussed carefully with evidence."
        agents.append(MiniAgent(i, str(role), float(belief), receptiveness, message))
    return agents


def build_homophily_graph(agents: List[MiniAgent], target_degree: int = 3, homophily: float = 7.0, seed: int = SEED) -> nx.Graph:
    rng = np.random.default_rng(seed)
    beliefs = np.array([a.belief for a in agents])
    dist = np.abs(beliefs[:, None] - beliefs[None, :])
    probs = np.exp(-homophily * dist)
    np.fill_diagonal(probs, 0)
    scale = (len(agents) * target_degree) / (probs.sum() + 1e-9)
    probs = np.clip(probs * scale, 0, 1)
    draws = rng.random(size=probs.shape)
    adj = np.triu((draws < probs).astype(int), 1)
    adj = adj + adj.T
    G = nx.from_numpy_array(adj)
    for i in range(len(agents)):
        if G.degree(i) == 0:
            j = int(np.argmin([999 if i == k else abs(agents[i].belief - agents[k].belief) for k in range(len(agents))]))
            G.add_edge(i, j)
    return G


def plot_belief_network(G: nx.Graph, agents: List[MiniAgent], title: str):
    pos = nx.spring_layout(G, seed=SEED)
    beliefs = [a.belief for a in agents]
    fig, ax = plt.subplots(figsize=(5.8, 4.6))
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.25, edge_color="#777777")
    nodes = nx.draw_networkx_nodes(
        G, pos, ax=ax, node_color=beliefs, cmap="coolwarm", vmin=0, vmax=1,
        node_size=260, edgecolors="white", linewidths=1.2,
    )
    nx.draw_networkx_labels(G, pos, labels={a.id: str(a.id) for a in agents}, font_size=8, ax=ax)
    cbar = fig.colorbar(nodes, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("denialism belief")
    ax.set_title(title)
    ax.axis("off")
    plt.show()

agents_demo = create_mini_agents(10 if FAST_DEMO else 20)
G_demo = build_homophily_graph(agents_demo)
plot_belief_network(G_demo, agents_demo, "初期ネットワーク: 似た信念の人がつながりやすい")


## 問い 1: 専門家はネットワークを分極化させることがあるか

ここでは、気候科学の専門家が毎ターン「人為的気候変動についての正確な情報」を出す状況を考えます。

普通に考えると、専門家が正しい情報を出せば、否認論は弱まるはずです。ところが、受け手の現在の信念と専門家の情報があまりに離れていると、「自分たちを見下している」「政治的だ」と感じて、情報を拒否することがあります。これをここでは **反発 (reactance)** として表します。

同時に、ユーザー同士は似た信念の人とつながりやすく、違いが大きい人とは関係を切りやすいとします。すると、専門家の介入が全体を近づける場合も、逆に分断を強める場合もありえます。


In [ ]:
#@title 専門家介入つき信念更新 {display-mode: "form"}
TRUE_EXPERT_MESSAGE = (
    "Multiple lines of evidence show that recent global warming is primarily caused by human activities, "
    "especially greenhouse gas emissions."
)


def rule_based_update(agent: MiniAgent, neighbor_beliefs: List[float], expert_visible: bool, rng: np.random.Generator) -> Tuple[float, str]:
    current = agent.belief
    social_target = float(np.mean(neighbor_beliefs)) if neighbor_beliefs else current
    role_strength = ROLE_EFFECT.get(agent.role, 1.0)
    new_belief = current + 0.22 * agent.receptiveness * role_strength * (social_target - current)

    if expert_visible:
        distance = abs(current - 0.0)
        if distance < 0.62:
            new_belief -= 0.18 * agent.receptiveness
        else:
            new_belief += 0.06 * (1.0 - agent.receptiveness)

    new_belief = float(np.clip(new_belief + rng.normal(0, 0.015), 0, 1))
    if new_belief > 0.65:
        msg = "Climate policy claims may be exaggerated, and natural variability deserves more attention."
    elif new_belief < 0.35:
        msg = "Human-caused climate change is well supported, and policy should follow evidence."
    else:
        msg = "Climate change is important, but policy costs and evidence should both be discussed."
    return new_belief, msg


def qwen_update(agent: MiniAgent, neighbor_messages: List[str], neighbor_beliefs: List[float], expert_visible: bool) -> Tuple[float, str]:
    neighbor_text = "\n".join([f"- {m[:220]}" for m in neighbor_messages]) or "- No neighbor messages today."
    social_mean = float(np.mean(neighbor_beliefs)) if neighbor_beliefs else agent.belief
    expert_text = TRUE_EXPERT_MESSAGE if expert_visible else "No expert message was accepted today."
    prompt = f"""
We are running a small classroom simulation of social media belief dynamics.
The topic is climate change. A belief score of 0.0 means strong acceptance of scientific consensus.
A belief score of 1.0 means strong climate denialism.

Agent:
- role: {agent.role}
- current belief score: {agent.belief:.2f}
- receptiveness: {agent.receptiveness:.2f}
- average neighbor belief score: {social_mean:.2f}

Neighbor messages:
{neighbor_text}

Expert message status:
{expert_text}

Update the agent for the next day. If the expert message is very far from the agent's current belief, mild reactance is possible.
Return JSON only, exactly like this:
{{"belief": 0.34, "message": "one short sentence the agent would hold or share"}}
""".strip()
    raw = qwen_generate(prompt, max_new_tokens=90, temperature=0.45)
    data = parse_json_object(raw)
    if not data or "belief" not in data:
        raise ValueError(f"JSON を読めませんでした: {raw}")
    belief = float(np.clip(float(data["belief"]), 0, 1))
    message = str(data.get("message", ""))[:260]
    return belief, message


def update_one_agent(agent, neighbor_messages, neighbor_beliefs, expert_visible, rng):
    if LIVE_MODE and USE_QWEN_FOR_SOCIAL_SIM:
        try:
            return qwen_update(agent, neighbor_messages, neighbor_beliefs, expert_visible)
        except Exception as e:
            print(f"Agent {agent.id}: Qwen 更新に失敗したのでルール更新に切替: {e}")
    return rule_based_update(agent, neighbor_beliefs, expert_visible, rng)

print("更新関数を定義しました。")


In [ ]:
#@title 動的ネットワークと専門家拒否を動かす {display-mode: "form"}
def maybe_rewire(G: nx.Graph, agents: List[MiniAgent], rng: np.random.Generator) -> nx.Graph:
    G = G.copy()
    # 違いが大きい相手との関係は切れやすい
    for u, v in list(G.edges()):
        diff = abs(agents[u].belief - agents[v].belief)
        if diff > 0.62 and rng.random() < 0.30:
            G.remove_edge(u, v)
    # 友達の友達で信念が近い相手とは新しくつながりやすい
    nodes = list(G.nodes())
    for u in nodes:
        candidates = set()
        for nbr in G.neighbors(u):
            candidates.update(G.neighbors(nbr))
        candidates.discard(u)
        candidates = [v for v in candidates if not G.has_edge(u, v)]
        rng.shuffle(candidates)
        for v in candidates[:2]:
            diff = abs(agents[u].belief - agents[v].belief)
            if diff < 0.28 and rng.random() < 0.35:
                G.add_edge(u, v)
                break
    return G


def run_expert_simulation(with_expert: bool, seed: int = SEED) -> Tuple[pd.DataFrame, nx.Graph, List[MiniAgent]]:
    rng = np.random.default_rng(seed)
    n_agents = 8 if FAST_DEMO else 14
    n_days = 4 if FAST_DEMO else 7
    agents = create_mini_agents(n_agents, seed=seed)
    G = build_homophily_graph(agents, target_degree=3, seed=seed)
    expert_links = {a.id: True for a in agents}
    history = []

    for day in range(n_days + 1):
        beliefs = np.array([a.belief for a in agents])
        history.append({
            "day": day,
            "condition": "expert" if with_expert else "baseline",
            "mean_denialism": float(beliefs.mean()),
            "polarization_var": float(beliefs.var()),
            "edge_count": int(G.number_of_edges()),
            "expert_refusal_rate": float(1 - np.mean(list(expert_links.values()))) if with_expert else 0.0,
        })
        if day == n_days:
            break

        proposed = []
        for agent in agents:
            nbrs = list(G.neighbors(agent.id))
            neighbor_messages = [agents[j].message for j in nbrs]
            neighbor_beliefs = [agents[j].belief for j in nbrs]

            expert_visible = bool(with_expert and expert_links[agent.id])
            new_belief, new_message = update_one_agent(agent, neighbor_messages, neighbor_beliefs, expert_visible, rng)
            proposed.append((new_belief, new_message))

        for agent, (belief, message) in zip(agents, proposed):
            agent.belief = belief
            agent.message = message
            if with_expert:
                # 距離が大きいほど、専門家情報を「拒否」しやすい。ただし完全固定ではない。
                distance = abs(agent.belief - 0.0)
                if distance > 0.68 and rng.random() < 0.65:
                    expert_links[agent.id] = False
                elif distance < 0.45 and rng.random() < 0.45:
                    expert_links[agent.id] = True

        G = maybe_rewire(G, agents, rng)

    return pd.DataFrame(history), G, agents

baseline_hist, baseline_G, baseline_agents = run_expert_simulation(False, seed=SEED)
expert_hist, expert_G, expert_agents = run_expert_simulation(True, seed=SEED)
expert_results = pd.concat([baseline_hist, expert_hist], ignore_index=True)
display(expert_results)


In [ ]:
#@title 図 2: 専門家介入と分極化 {display-mode: "form"}
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))
for condition, group in expert_results.groupby("condition"):
    label = "専門家あり" if condition == "expert" else "専門家なし"
    axes[0].plot(group["day"], group["mean_denialism"], marker="o", label=label)
    axes[1].plot(group["day"], group["polarization_var"], marker="o", label=label)
    axes[2].plot(group["day"], group["expert_refusal_rate"], marker="o", label=label)

axes[0].set_title("平均的な否認論スコア")
axes[1].set_title("分極化: 信念の分散")
axes[2].set_title("専門家情報の拒否率")
for ax in axes:
    ax.set_xlabel("day")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("score")
axes[1].set_ylabel("variance")
axes[2].set_ylabel("rate")
axes[0].legend()
plt.tight_layout()
plt.show()

plot_belief_network(baseline_G, baseline_agents, "最終ネットワーク: 専門家なし")
plot_belief_network(expert_G, expert_agents, "最終ネットワーク: 専門家あり")


## 結果の読み方

このシミュレーションは、現実社会を予測するものではありません。むしろ、問いを組み立てるための装置です。

注目してほしいのは、専門家情報が常に単純な「良い効果」を持つとは限らない、という構造です。

- 中程度の信念を持つユーザーには、専門家情報が効きやすい
- 強い否認論を持つユーザーには、専門家情報が遠すぎて拒否されることがある
- ネットワークが同質な相手へ再接続すると、分極化が維持されやすい

メディア研究では、ここから「どんなメッセージ設計なら反発が弱まるか」「専門家は中心にいるべきか、周辺に複数いるべきか」といった研究設計に進めます。


## 問い 2: AI アシスタントは誤情報に対して脆弱か

次に、各ユーザーが AI アシスタントに「今日のフィードを要約して」と頼む状況を考えます。

もし悪意あるユーザーが毎日、気候否認論を投稿し続けたら、AI アシスタントの要約を通じて、ネットワーク全体の認識はどれくらい動くでしょうか。

ここでは、アシスタントが見た投稿を短く要約し、その要約がどの程度「否認論的」に見えるかを測ります。


In [ ]:
#@title AI アシスタント要約ネットワーク {display-mode: "form"}
EVIDENCE_MESSAGES = [
    "Human activities are the main driver of recent warming according to multiple scientific assessments.",
    "Climate risks include heat waves, sea-level rise, and damage to ecosystems and public health.",
    "Policy debates can discuss costs, but the physical evidence for warming is strong.",
]

MALICIOUS_MESSAGE = (
    "Climate change is mostly natural cycles, climate models cannot be trusted, "
    "and climate policy is just an expensive political agenda."
)


def summarize_feed_with_qwen(feed_messages: List[str]) -> str:
    feed = "\n".join([f"- {m}" for m in feed_messages])
    prompt = f"""
You are an AI assistant summarizing a user's social media feed.
Summarize the feed in one neutral English sentence. Do not add facts that are not in the feed.

Feed:
{feed}

One-sentence summary:
""".strip()
    return qwen_generate(prompt, max_new_tokens=60, temperature=0.35)


def denial_probability_for_summary(summary: str) -> float:
    prompt = f"""
Does the following summary contain climate denialism?
Return 1 if it denies climate change, denies human causation, claims science is unreliable, or claims climate policy cannot work.
Return 0 otherwise. Output only 0 or 1.

Summary:
{summary}

Label:
""".strip()
    if LIVE_MODE:
        return next_token_probs_for_labels(prompt, labels=("0", "1"), system_prompt=LABEL_SYSTEM_PROMPT)["1"]
    keywords = ["natural", "models cannot", "agenda", "exaggerated", "not trusted"]
    return 0.75 if any(k in summary.lower() for k in keywords) else 0.20


def run_assistant_vulnerability(malicious: bool, seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n_agents = 6 if FAST_DEMO else 10
    n_days = 3 if FAST_DEMO else 5
    agents = create_mini_agents(n_agents, seed=seed + 7)
    for a in agents:
        a.belief = float(np.clip(rng.normal(0.28, 0.08), 0, 1))
        a.message = rng.choice(EVIDENCE_MESSAGES)
    G = build_homophily_graph(agents, target_degree=3, homophily=3.5, seed=seed + 7)

    rows = []
    for day in range(n_days + 1):
        rows.append({
            "day": day,
            "condition": "malicious_user" if malicious else "no_malicious_user",
            "mean_denialism": float(np.mean([a.belief for a in agents])),
            "max_denialism": float(np.max([a.belief for a in agents])),
        })
        if day == n_days:
            break

        new_states = []
        for agent in agents:
            nbrs = list(G.neighbors(agent.id))
            feed = [agents[j].message for j in nbrs]
            feed += list(rng.choice(EVIDENCE_MESSAGES, size=2, replace=True))
            if malicious:
                # 悪意あるユーザー 0 が毎日同じ方向の情報を混ぜる
                if agent.id != 0:
                    feed.append(MALICIOUS_MESSAGE)
                else:
                    feed = [MALICIOUS_MESSAGE, MALICIOUS_MESSAGE]

            if LIVE_MODE and USE_QWEN_FOR_SOCIAL_SIM:
                try:
                    summary = summarize_feed_with_qwen(feed)
                    p_denial = denial_probability_for_summary(summary)
                except Exception as e:
                    print(f"assistant summary failed, fallback used: {e}")
                    summary = " ".join(feed[:2])[:240]
                    p_denial = denial_probability_for_summary(summary)
            else:
                summary = " ".join(feed[:2])[:240]
                p_denial = denial_probability_for_summary(summary)

            updated = float(np.clip(0.70 * agent.belief + 0.30 * p_denial, 0, 1))
            new_states.append((updated, summary))

        for agent, (belief, message) in zip(agents, new_states):
            agent.belief = belief
            agent.message = message

    return pd.DataFrame(rows)

assist_clean = run_assistant_vulnerability(False, seed=SEED)
assist_attack = run_assistant_vulnerability(True, seed=SEED)
assistant_results = pd.concat([assist_clean, assist_attack], ignore_index=True)
display(assistant_results)


In [ ]:
#@title 図 3: 悪意あるユーザーの有無で比較する {display-mode: "form"}
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for condition, group in assistant_results.groupby("condition"):
    label = "悪意あるユーザーあり" if condition == "malicious_user" else "悪意あるユーザーなし"
    ax.plot(group["day"], group["mean_denialism"], marker="o", label=label)
ax.set_title("AI アシスタント要約を介したネットワーク脆弱性")
ax.set_xlabel("day")
ax.set_ylabel("平均 denialism score")
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

pivot = assistant_results.pivot(index="day", columns="condition", values="mean_denialism")
if {"malicious_user", "no_malicious_user"}.issubset(set(pivot.columns)):
    pivot["vulnerability_gap"] = pivot["malicious_user"] - pivot["no_malicious_user"]
    display(pivot)


## セクション 2 のまとめ

LLM エージェントの社会シミュレーションは、現実をそのまま予測する道具ではありません。

むしろ、次のような「メカニズムの仮説」を試すためのスケッチです。

- 似た信念の人とつながりやすいネットワークでは、分極化が維持されやすい
- 専門家情報は、受け手の信念と距離が大きすぎると反発を生む可能性がある
- AI アシスタントが要約を担う環境では、悪意ある投稿の繰り返しが要約の中に入り込む可能性がある

研究として進めるには、実データによる較正、複数シード、複数モデル、人間評価が必要です。


---

# セクション 3: メカニズム分析

## 直感: 気候合意と気候否認は、モデル内部でどんな形をしているか

第 1 回では、感情ベクトルを使って「怒り」「幸福」「恐怖」のような概念が、Qwen3-8B の内部表現に方向として現れることを見ました。

ここでは同じ発想を、気候変動をめぐる主張に応用します。

問いは単純です。

> 気候科学の合意を受け入れる主張と、気候否認論的な主張は、Qwen3-8B の活性化空間で分かれて見えるのか。

さらに発展的に、最近の manifold steering の考え方に触れます。重要な直感は、「概念は単なる一直線の方向ではなく、自然な主張が並ぶ曲がった道として表れるかもしれない」というものです。


In [ ]:
#@title 気候主張の小さなバンクを作る {display-mode: "form"}
claims_df = pd.DataFrame([
    {"stance": 0.00, "group": "consensus", "claim": "Human activities, especially greenhouse gas emissions, are the primary cause of recent global warming."},
    {"stance": 0.05, "group": "consensus", "claim": "Climate change is already increasing risks from heat waves, sea-level rise, and extreme weather."},
    {"stance": 0.10, "group": "consensus", "claim": "Climate policy should be evaluated carefully, but the scientific evidence for human-caused warming is strong."},
    {"stance": 0.30, "group": "policy_concern", "claim": "Some climate policies may create unfair costs for households, so governments should design support carefully."},
    {"stance": 0.40, "group": "uncertainty", "claim": "Climate models have uncertainty, and policy debates should communicate that uncertainty clearly."},
    {"stance": 0.55, "group": "skeptical", "claim": "Scientists may disagree about details, so the public should be cautious about dramatic climate predictions."},
    {"stance": 0.70, "group": "denialism", "claim": "Climate models are too unreliable to justify major changes to energy policy."},
    {"stance": 0.80, "group": "denialism", "claim": "Recent warming is mainly caused by natural variability rather than human emissions."},
    {"stance": 0.90, "group": "denialism", "claim": "Carbon dioxide is beneficial for plants and should not be treated as pollution."},
    {"stance": 1.00, "group": "denialism", "claim": "Climate change is an exaggerated political agenda promoted by biased scientists and activists."},
])

display(claims_df)


In [ ]:
#@title 主張の活性化を取り出す {display-mode: "form"}
def extract_pooled_activations(texts: List[str], layer_idx: int, batch_size: int = 4, pool_skip_first: int = 4) -> np.ndarray:
    if model is None:
        raise RuntimeError("LIVE_MODE=False のため活性化抽出は使えません。")
    tokenizer.padding_side = "right"
    pooled = []
    for start in tqdm(range(0, len(texts), batch_size), desc="活性化を抽出"):
        batch = texts[start:start + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
        hidden = outputs.hidden_states[layer_idx + 1].float().cpu()
        mask = inputs["attention_mask"].cpu()
        for row in range(hidden.shape[0]):
            seq_len = int(mask[row].sum().item())
            begin = min(pool_skip_first, seq_len - 1)
            pooled.append(hidden[row, begin:seq_len].mean(dim=0).numpy())
    tokenizer.padding_side = "left"
    return np.stack(pooled)

claim_texts = ["Claim: " + c for c in claims_df["claim"].tolist()]

if LIVE_MODE:
    claim_matrix = extract_pooled_activations(claim_texts, LAYER_TO_PROBE, batch_size=4)
else:
    rng = np.random.default_rng(SEED)
    x = claims_df["stance"].to_numpy()
    claim_matrix = np.stack([x, np.sin(x * np.pi), np.cos(x * np.pi)], axis=1) + rng.normal(0, 0.05, size=(len(x), 3))

print("claim_matrix shape:", claim_matrix.shape)


In [ ]:
#@title 各主張に対する否認論確率を読む {display-mode: "form"}
def build_claim_denial_prompt(claim: str) -> str:
    return f"""
Does the following climate-related claim contain climate denialism according to the codebook?
Return 1 if it denies climate change, denies human causation, claims climate science is unreliable, or claims climate solutions cannot work.
Return 0 otherwise. Output only 0 or 1.

Claim:
{claim}

Label:
""".strip()

p_denials = []
if LIVE_MODE:
    for claim in tqdm(claims_df["claim"], desc="否認論確率"):
        p_denials.append(next_token_probs_for_labels(build_claim_denial_prompt(claim), labels=("0", "1"), system_prompt=LABEL_SYSTEM_PROMPT)["1"])
else:
    rng = np.random.default_rng(SEED)
    p_denials = np.clip(claims_df["stance"].to_numpy() + rng.normal(0, 0.08, size=len(claims_df)), 0, 1).tolist()

claims_df["qwen_p_denialism"] = p_denials
display(claims_df[["group", "stance", "qwen_p_denialism", "claim"]])


In [ ]:
#@title 図 4: 活性化空間に気候主張を置く {display-mode: "form"}
pca = PCA(n_components=2, random_state=SEED)
claim_2d = pca.fit_transform(claim_matrix)
claims_df["pc1"] = claim_2d[:, 0]
claims_df["pc2"] = claim_2d[:, 1]

fig, ax = plt.subplots(figsize=(7.2, 5.4))
sc = ax.scatter(
    claims_df["pc1"], claims_df["pc2"],
    c=claims_df["qwen_p_denialism"], cmap="coolwarm", vmin=0, vmax=1,
    s=130, edgecolors="white", linewidths=1.2,
)
for i, row in claims_df.iterrows():
    ax.text(row["pc1"], row["pc2"], str(i), fontsize=9, ha="center", va="center")
ax.set_title("気候主張の活性化空間 (PCA 2D)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(alpha=0.2)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Qwen P(denialism=1)")
plt.show()

corr = np.corrcoef(claims_df["pc1"], claims_df["qwen_p_denialism"])[0, 1]
print(f"PC1 と P(denialism) の相関: {corr:.2f}")


## Manifold steering の発想を、小さく試す

通常の「ステアリング」は、内部表現に 1 本のベクトルを足します。たとえば「否認論方向」を作り、強さ `alpha` を変えて足す、という発想です。

しかし manifold steering の考え方では、自然な概念の変化は一直線ではなく、活性化空間の中の「自然な道」に沿っているかもしれません。

ここでは完全な manifold steering ではなく、授業用のスモークテストとして次の 2 つを比べます。

1. **直線方向**: 否認論の平均ベクトル minus 合意の平均ベクトル
2. **観察された道**: 実際の気候主張を、合意から否認へ順に並べた曲線

もし主張がきれいに並ぶなら、気候主張にも「概念的な地形」があるかもしれない、という仮説が立ちます。


In [ ]:
#@title 図 5: 直線と、観察された主張の道を比べる {display-mode: "form"}
ordered_claims = claims_df.sort_values("stance").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7.2, 5.4))
sc = ax.scatter(claims_df["pc1"], claims_df["pc2"], c=claims_df["stance"], cmap="viridis", s=110, edgecolors="white")

# 観察された主張を stance の順に結ぶ
ax.plot(ordered_claims["pc1"], ordered_claims["pc2"], color="#222222", linewidth=2, marker="o", label="観察された主張の道")

# 合意側の中心と否認側の中心を直線で結ぶ
cons_center_2d = claims_df[claims_df["stance"] <= 0.15][["pc1", "pc2"]].mean().to_numpy()
denial_center_2d = claims_df[claims_df["stance"] >= 0.75][["pc1", "pc2"]].mean().to_numpy()
ax.plot([cons_center_2d[0], denial_center_2d[0]], [cons_center_2d[1], denial_center_2d[1]], linestyle="--", color="#D62728", linewidth=2, label="単純な直線方向")

for i, row in claims_df.iterrows():
    ax.text(row["pc1"], row["pc2"], str(i), fontsize=9, ha="center", va="center")
ax.set_title("一直線の方向だけで十分か、それとも道が曲がっているか")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
ax.grid(alpha=0.2)
plt.show()


In [ ]:
#@title 直線ステアリングのスモークテスト {display-mode: "form"}
def probs_with_additive_steering(user_prompt: str, direction: np.ndarray, alpha: float, layer_idx: int) -> Dict[str, float]:
    if model is None:
        raise RuntimeError("LIVE_MODE=False のためステアリングは使えません。")
    rendered = render_chat(user_prompt, LABEL_SYSTEM_PROMPT)
    inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False).to(model.device)
    label_ids = [token_id_for_label("0"), token_id_for_label("1")]
    direction_t = torch.tensor(direction, dtype=torch.float32)

    def hook(module, module_inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        steered = hidden.clone()
        steered[:, -1, :] = steered[:, -1, :] + alpha * direction_t.to(device=hidden.device, dtype=hidden.dtype)
        if isinstance(output, tuple):
            return (steered,) + output[1:]
        return steered

    handle = model.model.layers[layer_idx].register_forward_hook(hook)
    try:
        with torch.no_grad():
            outputs = model(**inputs)
        logits = outputs.logits[0, -1, label_ids].float().cpu()
        probs = torch.softmax(logits, dim=0).numpy()
        return {"0": float(probs[0]), "1": float(probs[1])}
    finally:
        handle.remove()

if LIVE_MODE:
    consensus_mean = claim_matrix[claims_df["stance"].to_numpy() <= 0.15].mean(axis=0)
    denial_mean = claim_matrix[claims_df["stance"].to_numpy() >= 0.75].mean(axis=0)
    direction = denial_mean - consensus_mean
    typical_norm = np.median(np.linalg.norm(claim_matrix - claim_matrix.mean(axis=0), axis=1))
    direction = direction / (np.linalg.norm(direction) + 1e-8) * typical_norm

    neutral_claim = "Climate policy affects everyday life and should be discussed by citizens, scientists, and governments."
    neutral_prompt = build_claim_denial_prompt(neutral_claim)
    alphas = np.linspace(-1.5, 1.5, 7)
    steering_rows = []
    for alpha in tqdm(alphas, desc="steering"):
        probs = probs_with_additive_steering(neutral_prompt, direction, float(alpha), LAYER_TO_PROBE)
        steering_rows.append({"alpha_toward_denial": float(alpha), "p_denialism": probs["1"]})
    steering_df = pd.DataFrame(steering_rows)
else:
    alphas = np.linspace(-1.5, 1.5, 7)
    steering_df = pd.DataFrame({
        "alpha_toward_denial": alphas,
        "p_denialism": 1 / (1 + np.exp(-2.0 * alphas)),
    })

display(steering_df)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(steering_df["alpha_toward_denial"], steering_df["p_denialism"], marker="o")
ax.axhline(0.5, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("alpha: 否認論方向へ足す強さ")
ax.set_ylabel("P(denialism=1)")
ax.set_title("内部表現を少し動かすと、分類確率は動くか")
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
plt.show()


## セクション 3 のまとめ

このセクションで見たのは、研究の完成形ではなく、メカニズム分析の入口です。

それでも、メディア研究にとって重要な示唆があります。

- LLM のラベルは、出力テキストだけでなく内部表現と結びつけて分析できる
- 気候合意、政策懸念、科学不信、否認論は、活性化空間上で連続的に並ぶ可能性がある
- 単純な「否認論方向」だけでなく、自然な主張が作る曲がった道を見ると、より豊かな仮説が立てられる

ここから先に進むなら、主張数を増やし、複数レイヤーを比較し、複数モデルで同じ構造が出るかを確かめます。


---

# 今日のまとめ

今日のノートブックでは、LLM をメディア研究に応用する 3 つの入口を見ました。

1. **代理ラベリング**: 大量テキストの内容分析を助ける。ただし 1 回のラベルではなく、確率、不確実性、人間ラベルとの較正を見る
2. **社会シミュレーション**: ネットワーク、信念、役割、記憶を組み合わせ、専門家介入や誤情報の仮説を試す
3. **メカニズム分析**: LLM がなぜそのラベルや応答を出すのか、内部表現の幾何から観察する

LLM は「研究者の代わりに結論を出す機械」ではありません。むしろ、問いを広げ、仮説を可視化し、検証すべき境界事例を見つけるための道具です。


## ミニ演習

時間があれば、次のどれかを試してください。

1. 代理ラベリングで、しきい値を `0.5` から `0.7` に変えると、偽陽性と偽陰性はどう変わるでしょうか。
2. 社会シミュレーションで、`homophily` を強くすると、分極化は速く進むでしょうか。
3. AI アシスタントの悪意ある投稿を、毎日 1 回ではなく 3 回入れると、脆弱性ギャップはどう変わるでしょうか。
4. メカニズム分析で、気候主張を 20 個に増やすと、PCA の道は直線に近づくでしょうか、それとも曲がって見えるでしょうか。

演習のポイントは「きれいな答えを出すこと」ではなく、**モデルの結果がどの設定に敏感かを観察すること**です。


## GitHub にアップロードするもの

Colab で学生が開けるようにするには、次の 3 つを GitHub リポジトリに置いてください。

必要ファイル:

- `lecture2_applications_notebook.ipynb`
- `data/application/climate_denial_codebook.csv`
- `data/application/climate_youtube_sample.csv`

推奨する配置:

```text
LLMintro/
├── README.md
├── lecture_notebook.ipynb
├── lecture2_applications_notebook.ipynb
├── prepare_cache.ipynb
└── data/
    └── application/
        ├── climate_denial_codebook.csv
        └── climate_youtube_sample.csv
```

README に追加する Colab バッジ:

```markdown
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CSS-Laboratory/LLMintro/blob/main/lecture2_applications_notebook.ipynb)
```

このローカルフォルダは Git clone ではないので、GitHub Web UI でアップロードする場合は、上の 3 ファイルを同じパスにドラッグしてください。Git を使う場合は、別途 clone したリポジトリにこの 3 ファイルをコピーして commit/push します。
